# Creating Embeddings

A **word embedding** is a dense vector that represents a word, learned so that words used in similar contexts end up with similar vectors. This is more useful than one-hot encoding, where every word is an equally distant, sparse vector with no notion of similarity.

In this notebook we train a tiny embedding model from scratch on a handful of sentences, so the whole mechanism is visible. We turn words into context pairs, train a one-hidden-layer neural network to predict context words, and then read the hidden-layer weights back out as the embeddings.

## Learning Objectives

At the end of this notebook, you should be able to:

- Construct training pairs of target and context words from raw sentences.
- Build and train a one-hidden-layer neural network that learns word embeddings.
- Extract the learned embeddings from the network weights.
- Visualise the embeddings in two dimensions and interpret the layout.

## Set up

In [ ]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm

# Drawing the embeddings
import matplotlib.pyplot as plt

# Deep learning:
from keras.models import Sequential
from keras.layers import Dense, Input

### Loading custom functions

In [ ]:
# We load custom functions from a python file
from extra.utility import text_preprocessing, create_unique_word_dict

### Read data

In [ ]:
# Reading the text from the input folder
texts = pd.read_csv("data/sample.csv")
texts = [x for x in texts["text"]]
texts

## Building the training data

The idea behind these embeddings is the **skip-gram** approach: from each target word we try to predict the words that appear near it. We slide a window of size 2 over each sentence and record every (target, context) pair. The model that learns to predict context from a target word is forced to give related words similar internal representations, and those representations are the embeddings.

Setting `window=2` to create pairs of 2 words occurring together in the dataset.

In [ ]:
# Defining the window for context
window = 2

# Creating a placeholder for the scanning of the word list
word_lists = []
all_text = []

- We go through all the sentences in the dataset. Preprocess the sentences (tokenize and remove stop words)
- We then create pair of words which occur near to each other in a window size equal 2.
  - A pair is created for two words say `a` and `b` like `a,b` and `b,a`

In [ ]:
for text in texts:
    # Cleaning the text
    text = text_preprocessing(text)

    # Appending to the all text list
    all_text += text

    # Creating a context dictionary
    for i, word in enumerate(text):
        for w in range(window):
            # Getting the context that is ahead by *window* words
            if i + 1 + w < len(text):
                word_lists.append([word] + [text[(i + 1 + w)]])
            # Getting the context that is behind by *window* words
            if i - w - 1 >= 0:
                word_lists.append([word] + [text[(i - w - 1)]])

In [ ]:
word_lists

Taking out all the unique words from our dataset

In [ ]:
unique_word_dict = create_unique_word_dict(all_text)

In [ ]:
unique_word_dict

In [ ]:
# Defining the number of features (unique words)
n_words = len(unique_word_dict)

# Getting all the unique words
words = list(unique_word_dict.keys())

In [ ]:
# Creating the X and Y matrices using one hot encoding
X = []
Y = []

for i, word_list in tqdm(enumerate(word_lists)):
    # Getting the indices
    main_word_index = unique_word_dict.get(word_list[0])
    context_word_index = unique_word_dict.get(word_list[1])

    # Creating the placeholders
    X_row = np.zeros(n_words)
    Y_row = np.zeros(n_words)

    # One hot encoding the main word
    X_row[main_word_index] = 1

    # One hot encoding the Y matrix words
    Y_row[context_word_index] = 1

    # Appending to the main matrices
    X.append(X_row)
    Y.append(Y_row)

In [ ]:
# converting lists to Numpy arrays
X = np.asarray(X)
Y = np.asarray(Y)

## Training the embedding model

We build a one-hidden-layer network. The input is a one-hot vector over the vocabulary, the hidden layer has just 2 units (this size is the embedding dimension), and the output is a softmax over the vocabulary predicting context words. After training, we discard the output layer: the weights of the hidden layer are the word vectors we want.

In [ ]:
# Defining the size of the embedding
embed_size = 2

We build a 1 layer Neural network, with an `input dimension = total number of unique words`, which represents our vocabulary. And the `output dimension = size of vocabulary` to get the embeddings for each word of same dimension.

In [ ]:
# Defining the neural network
model = Sequential()

# layers
model.add(Input(shape=(X.shape[1],)))
model.add(Dense(units=embed_size, activation="linear"))
model.add(Dense(units=Y.shape[1], activation="softmax"))


# Compiling the ANN
model.compile(optimizer="adam", loss="categorical_crossentropy")
print(model.summary())

In [ ]:
# Optimizing the network weights
model.fit(x=X, y=Y, batch_size=256, epochs=1000)

**Result:** the sample contains 12 short sentences with 21 unique words, which produce 84 target-context pairs. Because the hidden layer has 2 units, each of the 21 words is represented by a 2D vector after training.

We do not use the network's predictions directly. The training objective is only a means to push related words (for example `king`, `queen`, `prince`, `princess`) towards similar positions in the 2D space, which is what the scatter plot below shows. With so little data and a random initialisation, the exact layout changes from run to run, but words that share contexts tend to sit near each other.

After training the network, we extract the trained weights from the first layer of the neural network, which will be the embeddings for our vocabulary

In [ ]:
# The input layer
weights = model.get_weights()[0]

In [ ]:
# Creating a dictionary to store the embeddings in. The key is a unique word and
# the value is the numeric vector
embedding_dict = {}
for word in words:
    embedding_dict.update({word: weights[unique_word_dict.get(word)]})

## Visualising the embeddings

### 2D visualisation

In [ ]:
# Plotting the embeddings
plt.figure(figsize=(10, 10))
for word in list(unique_word_dict.keys()):
    coord = embedding_dict.get(word)
    plt.scatter(coord[0], coord[1])
    plt.annotate(word, (coord[0], coord[1]))

### 3D visualisation

We can also view the embeddings in 3D. Set `embed_size = 3` in the cell above and rerun the notebook, then the cell below draws an interactive 3D scatter with Plotly.

Drag to rotate the plot, and hover over any point to read its word and exact coordinates. This solves the main problem with a static 3D image, where depth is impossible to judge and you cannot tell where a point such as `man` actually sits.

In [ ]:
# Interactive 3D plot (needs embed_size = 3 above, then rerun the notebook)
if embed_size >= 3:
    import plotly.express as px

    words = list(embedding_dict.keys())
    coords = np.array([embedding_dict[w] for w in words])

    fig = px.scatter_3d(x=coords[:, 0], y=coords[:, 1], z=coords[:, 2], text=words)
    fig.update_traces(
        hovertemplate="%{text}<br>x=%{x:.2f} y=%{y:.2f} z=%{z:.2f}<extra></extra>"
    )
    fig.update_layout(
        title="Word embeddings in 3D (drag to rotate, hover for position)"
    )
    fig.show()
else:
    print(
        "Set embed_size = 3 in the cell above and rerun the notebook to see the interactive 3D plot."
    )

### Saving the embeddings

The cell below writes the embeddings to `embeddings_output/embedding.txt` in a readable form (`word: [vector]`), which is handy for a quick look but is not a format any tool expects.

> The interactive [TensorFlow Embedding Projector](http://projector.tensorflow.org/) needs two tab-separated files instead: a `vecs.tsv` of plain vectors and a `meta.tsv` of the matching words. You produce those in the optional notebook 5 by following its tutorial. In other words, this `.txt` dump and the projector's `.tsv` files are separate things, so the `.txt` here is not meant to be uploaded to the projector.

In [ ]:
# Saving the embedding vector to a txt file
try:
    os.mkdir(f"{os.getcwd()}//embeddings_output")
except Exception as e:
    print(f"Cannot create output folder: {e}")

In [ ]:
with open(f"{os.getcwd()}//embeddings_output//embedding.txt", "w") as f:
    for key, value in embedding_dict.items():
        try:
            f.write(f"{key}: {value}\n")
        except Exception as e:
            print(f"Cannot write word {key} to dict: {e}")

- These embeddings are later reused as an embedding layer in larger neural networks trained for NLP tasks.
- A larger embedding size (a longer vector per word) can capture more nuance, at the cost of more parameters to train.
- Try changing the embedding size to 3 and plotting in 3D, or keep it on the 2D scatter plot, and notice the differences.

### Your turn: measure word similarity

Embeddings place related words close together. Quantify that with cosine similarity, then try the classic analogy.

**Steps**
1. Write `cosine_similarity(a, b)` for two vectors using NumPy (dot product divided by the product of the norms).
2. Write `most_similar(word)` that returns the vocabulary words (other than `word`) with the highest cosine similarity, using `embedding_dict`.
3. Try `most_similar("king")`, then build the analogy vector `king - man + woman` and find its closest word.

**Hint:** use `np.dot` and `np.linalg.norm`. For the analogy, build the target vector first, then compare it to every word vector.

**Expected:** with only 12 sentences the result is rough, but related royalty or gender words should rank near the top.

In [ ]:
def cosine_similarity(a, b):
    # TODO 1: return the cosine similarity between vectors a and b
    return ...


def most_similar(word, top_n=3):
    # TODO 2: rank all other words by cosine similarity to `word`,
    # then return the top_n most similar words
    ...


print("Most similar to 'king':", most_similar("king"))

# TODO 3: build the analogy vector king - man + woman and find its closest word
analogy_vec = ...
...

### Your turn: explore the window and embedding size

The window size and embedding dimension change what the model learns. Experiment and observe.

**Steps**
1. Change `window` to `3` in the cell near the top, then rerun the notebook from there.
2. Compare the number of context pairs (`len(word_lists)`) to the value you had with `window = 2`.
3. Set `embed_size = 3`, rerun, and view the interactive 3D plot in the visualisation section.

**Hint:** the pair count grows as the window widens, because each word now sees more neighbours.

**Expected:** a larger window produces more training pairs; a larger embedding size gives each word a longer vector and a richer (but harder to plot) space.

In [ ]:
# After changing `window` above and rerunning, check the number of context pairs here.
print(f"window = {window}, number of context pairs = {len(word_lists)}")

## Summary

In this notebook you:

- Constructed target-context word pairs from raw sentences using a sliding window.
- Built and trained a one-hidden-layer neural network that learns word embeddings.
- Extracted the embeddings from the hidden-layer weights.
- Visualised the 21 word vectors in two dimensions and interpreted the layout.

## References & Further Reading

- [**The Illustrated Word2vec**](https://jalammar.github.io/illustrated-word2vec/): A visual explanation of how word embeddings are learned and what they represent.
- [**Keras: The Sequential model**](https://keras.io/api/models/sequential/): Reference for the model API used to build the network.